# Fig3 — Cell–Cell Communication

A–C: Per-phase CellChat network circles | D: Pathway information flow | E–F: Glutamate L-R pair dotplots | G: Hormone stage-split heatmap


In [ ]:
%%R
library(CellChat)
library(dplyr)
library(ggplot2)
library(ggrepel)
library(patchwork)
library(ComplexHeatmap)


In [ ]:
%%R
# ============================================================================
# Configuration
# ============================================================================
proj_dir <- Sys.getenv("PROJ_DIR", unset = "path/to/your/project")  # <-- MODIFY THIS
setwd(file.path(proj_dir, "data", "human", "output"))


In [ ]:
%%R
indir  <- "Analysis/ccc/all_cell/by_phase"
outdir <- "Visualization/Fig3"
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)


In [ ]:
%%R
# ============================================================================
# Shared colour palettes
# ============================================================================
col_type_2 <- c(
  "Astrocyte" = "#2E8B57", "CALB2+ GABA" = "#FF1493", "Cholinergic" = "#FFFF00",
  "Choroid_plexus_epithelial" = "#FF6347", "Endothelial_cell" = "#1E90FF",
  "Ependymal" = "#FF69B4", "Fibroblast" = "#D2B48C", "Glioblast" = "#90EE90",
  "Granule_neuron" = "#B0C4DE", "Hippocampal_Glu" = "#6A5ACD",
  "Hypothalamic_Glu" = "#5F9EA0", "L2/3-CUX2 Glu" = "#87CEEB",
  "L2/3-SATB2 Glu" = "#87CEFA", "L4-RORB Glu" = "#6495ED",
  "L5-BCL11B Glu" = "#4169E1", "L5/6-THEMIS Glu" = "#0000CD",
  "L5/6-TLE4 Glu" = "#ABC8E5", "LHX6+LAMP5+ GABA" = "#DC143C",
  "Macrophage" = "#A0522D", "Medium_spiny_neuron" = "#9400D3",
  "Microglia" = "#8B4513", "Midbrain-derived_GABA" = "#800080",
  "Neuroblast" = "#CAA2F4", "Neuronal_IPC" = "#FC9A9A",
  "Oligodendrocyte" = "#DC143C", "Oligodendrocyte_precursor" = "#32CD32",
  "Oligodendrocyte_precursor_committed" = "#ADFF2F", "Pericyte" = "#6495ED",
  "Purkinje_neuron" = "#BA55D3", "PVALB+ GABA" = "#CA81A9",
  "Radial_glia" = "#4682B4", "RELN+ GABA" = "#EF82B2",
  "Serotonergic" = "#FFA500", "SST+ GABA" = "#FF4500",
  "Thalamic_GABA" = "#9932CC", "Thalamic_Glu" = "#7B68EE",
  "Vascular_smooth_muscle_cell" = "#FF8C00", "VIP+ GABA" = "#C71585"
)


In [ ]:
%%R
color_phase <- c(
  "Prenatal" = "#3953A4",
  "Postnatal_developmental" = "#F89420",
  "Adult_and_aging" = "#BE1E2D"
)


In [ ]:
%%R
# ============================================================================
# Panels A–C: Per-phase network circle plots (interaction weights)
# ============================================================================
phases <- c("Prenatal", "Postnatal_developmental", "Adult_and_aging")
phase_labels <- c("Prenatal", "Postnatal Developmental", "Adult and Aging")


In [ ]:
%%R
for (i in seq_along(phases)) {
  ph  <- phases[i]
  rds <- file.path(indir, paste0("cellchat_", gsub(" ", "_", ph), "_ccc.rds"))
  if (!file.exists(rds)) {
    message("Skipping ", ph, ": RDS not found")
    next
  }
  cellchat <- readRDS(rds)
  pdf(file.path(outdir, paste0("CellChat_Network_Circle_weights_", ph, ".pdf")),
      width = 6, height = 6)
  groupSize  <- as.numeric(table(cellchat@idents))
  cell_order <- levels(cellchat@idents)
  color.use  <- col_type_2[cell_order]
  netVisual_circle(
    cellchat@net$weight,
    vertex.weight = groupSize, weight.scale = TRUE,
    label.edge = FALSE, edge.width.max = 4,
    color.use = color.use, title.name = "Interaction weights/strength"
  )
  dev.off()
  message("Panel ", LETTERS[i], " saved: ", ph)
}


In [ ]:
%%R
# ============================================================================
# Panel D: Pathway information flow — 3-panel bar chart
# ============================================================================
cellchat_list <- lapply(phases, function(ph) {
  rds <- file.path(indir, paste0("cellchat_", gsub(" ", "_", ph), "_ccc.rds"))
  if (file.exists(rds)) readRDS(rds) else NULL
})
names(cellchat_list) <- phases
cellchat_list <- cellchat_list[!sapply(cellchat_list, is.null)]


In [ ]:
%%R
# Rebuild empty @data for mergeCellChat compatibility
gene_names <- rownames(cellchat_list[[1]]@data.signaling)
for (nm in names(cellchat_list)) {
  cellchat_list[[nm]]@data <- matrix(
    NA, nrow = length(gene_names), ncol = ncol(cellchat_list[[nm]]@meta),
    dimnames = list(gene_names, colnames(cellchat_list[[nm]]@meta))
  )
}
cellchat_merged <- mergeCellChat(cellchat_list, add.names = names(cellchat_list))


In [ ]:
%%R
rank_data <- rankNet(cellchat_merged, measure = "weight", mode = "comparison",
                     comparison = c(1, 2, 3), do.stat = TRUE,
                     cutoff.pvalue = 0.05, return.data = TRUE)


In [ ]:
%%R
pathway_weight <- rank_data$signaling.contribution %>%
  group_by(name) %>%
  summarise(mean_weight = mean(contribution.scaled, na.rm = TRUE)) %>%
  arrange(desc(mean_weight))


In [ ]:
%%R
top_n <- 20
high_weight_pathways <- pathway_weight$name[1:top_n]


In [ ]:
%%R
plot_df <- rank_data$signaling.contribution %>%
  rename(pathway = name, weight = contribution,
         weight.scale = contribution.scaled, dataset = group) %>%
  filter(pathway %in% high_weight_pathways) %>%
  mutate(dataset = factor(dataset, levels = phases)) %>%
  group_by(pathway) %>%
  mutate(total_weight = sum(weight.scale, na.rm = TRUE)) %>%
  ungroup() %>%
  mutate(pathway = factor(pathway, levels = unique(pathway[order(-total_weight)])))


In [ ]:
%%R
p_d <- ggplot(plot_df, aes(x = weight, y = pathway, fill = dataset)) +
  geom_col(position = "dodge", width = 0.8) +
  scale_fill_manual(values = color_phase) +
  facet_wrap(~dataset, ncol = 3, scales = "free_x") +
  scale_y_discrete(limits = rev) +
  theme_bw() +
  theme(
    axis.text.y = element_text(size = 8),
    axis.text.x = element_text(size = 10),
    panel.grid = element_blank(),
    legend.position = "none",
    panel.border = element_rect(color = "black", linewidth = 0.6, fill = NA),
    axis.line = element_line(color = "black"),
    strip.text = element_blank(),
    strip.background = element_blank()
  ) +
  labs(x = "Information Flow", y = "Signaling Pathway")


In [ ]:
%%R
ggsave(file.path(outdir, "interactions_by_phase_pathway_weight_3panel.pdf"),
       p_d, width = 6, height = 6)
message("Panel D saved")


In [ ]:
%%R
# ============================================================================
# Panel E: Glutamate L-R pair dotplot across phases
# ============================================================================
df_net <- subsetCommunication(cellchat_merged, signaling = "Glutamate")


In [ ]:
%%R
lr_all <- bind_rows(lapply(names(df_net), function(phase) {
  df <- df_net[[phase]]; df$phase <- phase; df
}))


In [ ]:
%%R
lr_mean <- lr_all %>%
  group_by(phase, interaction_name) %>%
  summarise(mean_prob = mean(prob, na.rm = TRUE), .groups = "drop")


In [ ]:
%%R
unique_interactions <- unique(lr_mean$interaction_name)
lr_plot <- lr_mean %>%
  mutate(
    x_pos = match(interaction_name, sort(unique_interactions)),
    phase = factor(phase, levels = phases,
                   labels = c("Prenatal", "Postnatal Developmental", "Adult and Aging"))
  )


In [ ]:
%%R
top_lr <- lr_plot %>%
  group_by(phase) %>% arrange(desc(mean_prob)) %>% slice_head(n = 6) %>%
  ungroup() %>%
  mutate(label_name = sub("^Glutamate-Glu-", "", interaction_name))


In [ ]:
%%R
color_phase_labeled <- c(
  "Prenatal" = "#3953A4", "Postnatal Developmental" = "#F89420",
  "Adult and Aging" = "#BE1E2D"
)


In [ ]:
%%R
p_e <- ggplot(lr_plot, aes(x = x_pos, y = mean_prob)) +
  geom_point(aes(color = phase), size = 2) +
  geom_text_repel(
    data = top_lr, aes(label = label_name), size = 3, color = "black",
    max.overlaps = Inf, box.padding = 0.3, segment.size = 0.2,
    segment.color = "gray50"
  ) +
  scale_color_manual(values = color_phase_labeled) +
  facet_wrap(~phase, nrow = 1, scales = "free") +
  labs(x = "Ligand-Receptor Pair", y = "Mean Communication Probability",
       color = "Developmental Phase") +
  theme_bw() +
  theme(
    axis.text.x = element_blank(), axis.ticks.x = element_blank(),
    strip.text = element_text(size = 11, face = "bold"),
    strip.background = element_rect(fill = "white", color = "black"),
    legend.position = "top", panel.grid.minor = element_blank(),
    panel.grid.major.x = element_blank()
  )


In [ ]:
%%R
ggsave(file.path(outdir, "Glutamate_LR_pair_dotplot.pdf"), p_e, width = 8, height = 6)
message("Panel E saved")


In [ ]:
%%R
# ============================================================================
# Panel F: Cell-pair × L-R pair dotplot for top Glutamate interactions
# ============================================================================
pairLR <- data.frame(interaction_name = unique(top_lr$interaction_name))


In [ ]:
%%R
extract_phase_data <- function(cellchat_obj, phase_label, phase_color) {
  df_net <- subsetCommunication(cellchat_obj, signaling = "Glutamate")
  cell_pairs <- df_net %>%
    filter(interaction_name %in% top_lr$interaction_name, pval < 0.05) %>%
    arrange(desc(prob)) %>% head(10)
  sources_use <- unique(cell_pairs$source)
  targets_use <- unique(cell_pairs$target)
  subsetCommunication(cellchat_obj, pairLR.use = pairLR,
                      sources.use = sources_use, targets.use = targets_use) %>%
    filter(pval < 0.05) %>%
    mutate(
      phase = phase_label, phase_color = phase_color,
      cell_pair = paste(source, "->", target),
      label_name = sub("^Glutamate-Glu-", "", interaction_name)
    ) %>%
    select(phase, label_name, cell_pair, prob, pval)
}


In [ ]:
%%R
data_combined <- bind_rows(
  extract_phase_data(cellchat_list[["Prenatal"]], "Prenatal", "#3953A4"),
  extract_phase_data(cellchat_list[["Postnatal_developmental"]],
                     "Postnatal Developmental", "#F89420"),
  extract_phase_data(cellchat_list[["Adult_and_aging"]],
                     "Adult and Aging", "#BE1E2D")
) %>%
  mutate(
    phase = factor(phase, levels = c("Adult and Aging",
                    "Postnatal Developmental", "Prenatal")),
    cell_pair = factor(cell_pair, levels = unique(cell_pair[order(phase)]))
  )


In [ ]:
%%R
p_f <- ggplot(data_combined, aes(x = label_name, y = cell_pair)) +
  geom_point(aes(fill = prob), shape = 21, size = 4, color = "black", stroke = 1) +
  scale_fill_distiller(palette = "RdYlBu", direction = -1,
                       name = "Communication Probability") +
  labs(x = "", y = "") +
  theme_bw() +
  theme(
    axis.text.y = element_text(size = 12, face = "bold", hjust = 1),
    axis.text.x = element_text(angle = 90, hjust = 1, size = 10, face = "bold"),
    legend.title = element_text(size = 10, face = "bold"),
    legend.text = element_text(size = 9),
    legend.key = element_rect(fill = "white"),
    legend.position = "right",
    panel.grid = element_blank()
  )


In [ ]:
%%R
ggsave(file.path(outdir, "Glutamate_LR_pair_ctpair_dotplot3.pdf"),
       p_f, width = 11, height = 10)
message("Panel F saved")


In [ ]:
%%R
message("Fig3 complete. All panels saved to: ", outdir)


In [ ]:
%%R
  hdf <- read.csv(stage_file)
  # Expected columns: Pair, Hormone, Prenatal, Postnatal_developmental, Adult_aged


In [ ]:
%%R
  stage_groups <- c("Prenatal", "Postnatal_developmental", "Adult_aged")
  n_groups <- length(stage_groups)


In [ ]:
%%R
  pair_order   <- sort(unique(hdf$Pair))
  hormone_order <- sort(unique(hdf$Hormone))


In [ ]:
%%R
  # Pivot to wide matrix-style cell_data: (pair × hormone) → [3 values]
  hdf_long <- hdf %>%
    pivot_longer(all_of(stage_groups), names_to = "StageGroup", values_to = "Strength")


In [ ]:
%%R
  # Build data frame for ggplot
  plot_df <- expand.grid(
    Pair = pair_order, Hormone = hormone_order,
    StageGroup = stage_groups, stringsAsFactors = FALSE
  ) %>%
    left_join(hdf_long, by = c("Pair", "Hormone", "StageGroup")) %>%
    mutate(
      Pair = factor(Pair, levels = rev(pair_order)),
      Hormone = factor(Hormone, levels = hormone_order),
      StageGroup = factor(StageGroup, levels = stage_groups)
    )


In [ ]:
%%R
  vmin <- min(plot_df$Strength, na.rm = TRUE)
  vmax <- max(plot_df$Strength, na.rm = TRUE)


In [ ]:
%%R
  p_g <- ggplot(plot_df, aes(x = Hormone, y = Pair)) +
    geom_tile(aes(fill = Strength, alpha = StageGroup), color = "grey90", linewidth = 0.1) +
    scale_fill_gradient(low = "white", high = "darkred", na.value = "white",
                        limits = c(vmin, vmax)) +
    scale_alpha_manual(values = c("Prenatal" = 1, "Postnatal_developmental" = 0.7,
                                  "Adult_aged" = 0.4)) +
    labs(x = "Hormone", y = "Producer → Receiver",
         fill = "Mean communication\nstrength") +
    theme_minimal() +
    theme(
      axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 7),
      axis.text.y = element_text(size = 6.5),
      legend.position = "right",
      panel.grid = element_blank()
    )


In [ ]:
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

proj_dir = os.environ.get('PROJ_DIR', 'path/to/your/project')
data_dir = os.path.join(proj_dir, 'data', 'human', 'output')
ccc_dir  = os.path.join(data_dir, 'Analysis', 'ccc', 'all_cell', 'by_stage')
out_dir  = os.path.join(data_dir, 'Visualization', 'Fig3')
os.makedirs(out_dir, exist_ok=True)

stage_file = os.path.join(ccc_dir, 'Hormone_ProducerReceiver_stageSplit_data.csv')

if os.path.exists(stage_file):
    plot_df = pd.read_csv(stage_file)
    stage_labels = ["Prenatal", "Postnatal developmental", "Adult & aged"]
    n_groups = len(stage_labels)

    pair_order = sorted(plot_df["Pair"].unique())
    hormone_order = sorted(plot_df["Hormone"].unique())

    cell_data = {}
    all_vals = []
    for _, row in plot_df.iterrows():
        p, h = row["Pair"], row["Hormone"]
        vals = [row.get(c, np.nan) for c in ["Prenatal", "Postnatal_developmental", "Adult_aged"]]
        cell_data[(p, h)] = vals
        for v in vals:
            if pd.notna(v):
                all_vals.append(v)

    pair_active = {}
    for p in pair_order:
        active = []
        for gi in range(n_groups):
            for h in hormone_order:
                key = (p, h)
                if key in cell_data and not np.isnan(cell_data[key][gi]):
                    active.append(gi)
                    break
        if active:
            pair_active[p] = active

    active_pairs = [p for p in pair_order if p in pair_active]
    gap = 0.02
    pair_h = 1.0
    total_h = len(active_pairs) * pair_h
    n_cols = len(hormone_order)

    vmin, vmax = min(all_vals), max(all_vals)
    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.cm.Reds

    fig, ax = plt.subplots(figsize=(12, 5))
    for pi, p in enumerate(active_pairs):
        y_base = pi * pair_h
        n_act = len(pair_active[p])
        sub_h = pair_h / n_act
        active_groups = pair_active[p]
        for ci, h in enumerate(hormone_order):
            key = (p, h)
            if key in cell_data:
                stage_vals = cell_data[key]
                for si, gi in enumerate(active_groups):
                    val = stage_vals[gi]
                    y_top = y_base + si * sub_h
                    color = cmap(norm(val)) if not np.isnan(val) else "white"
                    rect = mpatches.Rectangle(
                        (ci + gap, y_top + gap), 1 - 2*gap, sub_h - 2*gap,
                        linewidth=0, edgecolor="none", facecolor=color)
                    ax.add_patch(rect)

    for i in range(len(active_pairs) + 1):
        ax.axhline(y=i, color="grey", linewidth=0.1)
    for j in range(n_cols + 1):
        ax.axvline(x=j, color="grey", linewidth=0.1)

    ax.set_xlim(0, n_cols)
    ax.set_ylim(total_h, 0)
    ax.set_xticks(np.arange(n_cols) + 0.5)
    ax.set_xticklabels(hormone_order, rotation=90, fontsize=7)
    ax.set_yticks([pi + 0.5 for pi in range(len(active_pairs))])
    ax.set_yticklabels(active_pairs, fontsize=6.5)
    ax.set_xlabel("Hormone")
    ax.set_ylabel("Producer -> Receiver")

    sm = ScalarMappable(norm=norm, cmap=cmap)
    cbar = plt.colorbar(sm, ax=ax, fraction=0.02, pad=0.04)
    cbar.set_label("Mean communication strength")
    plt.grid(False)
    plt.tight_layout()

    out_path = os.path.join(out_dir, "Hormone_ProducerReceiver_heatmap_3_stageSplit.pdf")
    fig.savefig(out_path, dpi=600, bbox_inches="tight")
    plt.close()
    print(f"Panel G saved: {out_path}")
else:
    print(f"Stage-split data not found: {stage_file}")
